In [1]:
import pandas as pd

## Exploring Plant Info

In [15]:
plant_info_path = "../data/august_generator2026.xlsx"
plant_info = pd.read_excel(plant_info_path, skiprows=2)

In [16]:
plant_info.shape

(28379, 37)

In [18]:
import geopandas as gpd
from shapely.geometry import Point

florida_bounds_path = "../data/florida_state_boundary.geojson"
gdf = gpd.read_file(florida_bounds_path)

In [20]:
florida_polygon = gdf.geometry.iloc[0]

In [21]:
florida_polygon

In [23]:
def check_in_florida(row):
  # checks if power plant is in florida

  lat, long = row["Latitude"], row["Longitude"]
  point = Point(long, lat)
  is_in_fl = florida_polygon.contains(point)
  return {"in_florida": is_in_fl}

In [27]:
in_florida = plant_info.apply(check_in_florida, axis=1, result_type="expand")

In [32]:
plant_info_w_loc = plant_info.join(in_florida)

In [33]:
filtered_plant_info = plant_info_w_loc[plant_info_w_loc["in_florida"] == True]

In [34]:
filtered_plant_info.shape

(749, 38)

In [37]:
no_dups_florida_plants = filtered_plant_info.drop_duplicates(subset=["Latitude", "Longitude"])

In [38]:
no_dups_florida_plants

,Entity ID,Entity Name,Plant ID,Plant Name,Google Map,Bing Map,Plant State,County,Balancing Authority Code,Sector,...,Planned Uprate Year,Planned Uprate Month,Planned Uprate of Summer Capacity (MW),Planned Repower Year,Planned Repower Month,Other Modification Year,Other Modification Month,Latitude,Longitude,in_florida
289,21554,Seminole Electric Cooperative Inc,136.0,Seminole (FL),Map,Map,FL,Putnam,SEC,Electric Utility,...,,,,,,,,29.733056,-81.63278,True
954,14610,Orlando Utilities Comm,564.0,Stanton Energy Center,Map,Map,FL,Orange,FMPP,Electric Utility,...,,,,,,,,28.482200,-81.16780,True
987,6452,Florida Power & Light Co,609.0,Cape Canaveral,Map,Map,FL,Brevard,FPL,Electric Utility,...,,,,,,,,28.469800,-80.76440,True
991,6452,Florida Power & Light Co,612.0,Fort Myers,Map,Map,FL,Lee,FPL,Electric Utility,...,,,,,,,,26.696700,-81.78310,True
1005,6452,Florida Power & Light Co,613.0,Lauderdale,Map,Map,FL,Broward,FPL,Electric Utility,...,,,,,,,,26.068600,-80.19840,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27756,6455,"Duke Energy Florida, LLC",69354.0,Rattler Solar Plant,Map,Map,FL,Hernando,FPC,Electric Utility,...,,,,,,,,28.634293,-82.28870,True
27757,6455,"Duke Energy Florida, LLC",69355.0,Half Moon Solar Plant,Map,Map,FL,Sumter,FPC,Electric Utility,...,,,,,,,,28.955619,-82.15959,True
27958,6455,"Duke Energy Florida, LLC",69627.0,Jumper Creek Solar Plant,Map,Map,FL,Sumter,FPC,Electric Utility,...,,,,,,,,28.954544,-82.17652,True
28254,60281,"Altus Power America Management, LLC",70179.0,"FL Solar D1, LLC",Map,Map,FL,Gadsden,FMPP,IPP Non-CHP,...,,,,,,,,30.654933,-84.42836,True


## Exploring Incident Reports

In [12]:
incident_reports_path = "../data/osha_florida_data.csv"
ir = pd.read_csv(plant_info_path, on_bad_lines="skip") # we need to skip bad lines because line 137 has " 12" deep " instead of " 12 inches deep "

In [11]:
ir

,ID,UPA,EventDate,Employer,Address1,Address2,city,state,Zip,Latitude,...,Nature,NatureTitle,Part_of_Body,Part_of_Body_Title,Event,EventTitle,Source,sourceTitle,Secondary_Source,Secondary_Source_Title
0,2025098920,2345433,2025-09-05,Carpenter Company,5100 South Frontage Road,NaN,LAKELAND,FLORIDA,33815.0,28.03,...,1220,"Amputations, avulsions, enucleations unspeci...",4421,"Finger or thumb tip(s), nail(s)",652,Struck by running powered equipment during m...,3519,"Extruding, injecting, forming, molding machin...",443.0,"Valves, nozzles"
1,2025098926,2345293,2025-09-06,ASAP Installations LLC,Buck Rd and little Econ River,NaN,ORLANDO,FLORIDA,32817.0,28.34,...,124,Fractures,520,Leg(s) unspecified,2533,Pedestrian struck by vehicle backing up in no...,3221,Mini excavators,991.0,Secondary source not applicable
2,2025098961,2345663,2025-09-08,Elro Manufacturing LLC,90 South Bradshaw Road,NaN,APOPKA,FLORIDA,32703.0,28.67,...,1220,"Amputations, avulsions, enucleations unspeci...",4421,"Finger or thumb tip(s), nail(s)",653,"Caught, entangled in running powered equipmen...",3520,"Bending, rolling, shaping, stamping machinery...",991.0,Secondary source not applicable
3,2025098979,2346886,2025-09-08,Manheim Fort Myers,2100 Rockfill Road,NaN,FORT MYERS,FLORIDA,33916.0,26.64,...,138,Multiple surface and flesh wounds,10,Head unspecified,4319,Fall on same level n.e.c.,6624,Other constructed surface,4722.0,"Gates, hatches vehicle and machine cargo"
4,2025098980,2346342,2025-09-08,"United Land Services Operating, LLC",Reverie Dr North,NaN,PALM COAST,FLORIDA,32137.0,29.56,...,160,"Effects of poison, toxic, or allergenic expos...",6,BODY SYSTEMS,6212,"Venomous animal bite except bugs""""",5141,Venomous snakes,991.0,Secondary source not applicable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9976,2016076124,1111104,2016-07-08,Miles of Smiles Rides Inc.,7825 54th ave n,NaN,SAINT PETERSBURG,FLORIDA,33705.0,27.82,...,111,Fractures,9999,Nonclassifiable,4336,Other fall to lower level 26 to 30 feet,9999,Nonclassifiable,NaN,NaN
9977,2016076136,1110639,2016-07-08,Orion Marine Group,5430 West Tyson Avenue,NaN,TAMPA,FLORIDA,33611.0,27.88,...,1972,"Soreness, pain, hurt-nonspecified injury",52,Ankle(s),4321,Fall through surface or existing opening less ...,6640,"Sidewalks, paths, outdoor walkways, unspecified",NaN,NaN
9978,2016076167,1110844,2016-07-08,"PGT Industries, Inc.",1070 Technology Dr.,NaN,NOKOMIS,FLORIDA,34275.0,27.14,...,1720,"Effects of heat and light, unspecified",6,BODY SYSTEMS,531,Exposure to environmental heat,9262,Heat-environmental,NaN,NaN
9979,2016076178,1110764,2016-07-09,Walt Disney World Parks and Resorts,200 Epcot Center,NaN,LAKE BUENA VISTA,FLORIDA,32830.0,28.38,...,1972,"Soreness, pain, hurt-nonspecified injury",43,Wrist(s),4213,Fall on same level due to tripping over an obj...,660,"Floors, walkways, ground surfaces, unspecified",40.0,"Parts and materials, unspecified"
